# 02 - Bayesian Change Point Modeling

Fits three variants of the change-point model defined in `src/modeling/change_point_model.py`
and compares them explicitly rather than assuming a single model is correct:

* `null` - no change point (baseline).
* `mean_shift` - the mean daily log return shifts at an unknown point `tau`.
* `mean_vol_shift` - both mean *and* volatility shift at `tau`.

`tau` is modeled as continuous with a sigmoid relaxation of the switch function, which
lets PyMC sample the whole model with NUTS instead of falling back to a poorly-mixing
compound Metropolis+NUTS step (see the module docstring for why the original
`DiscreteUniform` version produced unreliable diagnostics).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import MODEL_CONFIG, PATHS
from src.modeling.change_point_model import (
    compare_models,
    plot_posterior_mean_comparison,
    plot_price_with_change_point,
    plot_trace,
    run_single_model,
)

PATHS.ensure_dirs()
df = pd.read_csv(PATHS.processed_prices_csv, parse_dates=["Date"]).dropna(subset=["Log_Return"])
df.tail()

## Fit and compare all three model variants (WAIC/PSIS-LOO)

In [ ]:
results = {
    variant: run_single_model(df["Log_Return"].values, df["Date"], variant, MODEL_CONFIG)
    for variant in ("null", "mean_shift", "mean_vol_shift")
}
comparison = compare_models(results)
comparison

The top row of `comparison` (highest `elpd_loo`) is the model best supported by the
data. If a change-point variant doesn't clearly beat `null`, that is itself an
important, reportable finding - not a reason to discard the comparison.

In [ ]:
best_variant = comparison.index[0]
best = results[best_variant]
print("Best-supported model:", best_variant)
print("Diagnostics:", best.diagnostics)
best.summary

In [ ]:
if best_variant != "null":
    print("Change point date:", best.change_date)
    best.summary.to_csv(PATHS.trace_summary_csv)
    plot_trace(best.idata, list(best.summary.index), PATHS.figures / "change_point_traceplot.png")
    if "mu1" in best.summary.index:
        plot_posterior_mean_comparison(best.idata, PATHS.figures / "posterior_mu_comparison.png")
    plot_price_with_change_point(df, best.change_date, None, PATHS.figures / "price_with_change_point.png")
else:
    print("The null (no change point) model is best-supported for this window of data.")

## Next step

See `03_dashboard_logic.ipynb` for how the detected change point is aligned with the
curated events dataset and turned into an analyst-readable narrative, and
`pipelines/run_pipeline.py` to run this entire notebook's logic end-to-end as a
reproducible, logged CLI run.